# CLIP4CAD-GFA v4.8 Training

**Gap-Closing Codebook Architecture with Shared Fusion**

## Key Insight
InfoNCE only optimizes **relative ranking**, leaving a gap between modalities. v4.8 closes this gap **directly** with L_ATP (Align True Pairs), and uses a **SharedFusionNetwork** with the same weights for all modalities.

## Architecture Changes from v4.4
| Aspect | v4.4 | v4.8 |
|--------|------|------|
| Self-grounding path | Yes (complex) | **Eliminated** |
| Curriculum learning | Yes (hints 90%→0%) | **No** |
| Number of paths | 2 (guided + self) | **1** (single path) |
| Loss terms | 7 | **6** (well-motivated) |
| Codebook | No | **Yes** (256 shared codes) |
| Fusion | Separate proj_heads | **SharedFusionNetwork** |
| Gap handling | Work around it | **Close it directly** |

## Loss Terms
| Loss | Purpose | Weight |
|------|---------|--------|
| L_contrastive | 3-way InfoNCE (preserves retrieval) | 1.0 |
| L_ATP | Align True Pairs (closes gap) | 0.5 |
| L_CU | Centroid Uniformity (prevents collapse) | 0.3 |
| L_code | KL code alignment | 0.3 |
| L_diversity | Codebook utilization | 0.1 |
| L_hard_neg | Hard negative mining (Stage 2) | 0.3 |

## Training Stages
- **Stage 1 (Epochs 1-20)**: Gap closing + code learning
- **Stage 2 (Epochs 21-35)**: Hard negative refinement

## Expected Metrics
- Modality Gap: **→ 0** (direct alignment)
- True Pair Cosine: **→ 1.0**
- Code Diversity: **> 0.8** (using most codes)

In [1]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Memory: 25.8 GB


In [ ]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

# PC and Text files
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/gfa_v4_8")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 512           # Larger batch - simpler model
STAGE1_EPOCHS = 20         # Gap closing + code learning
STAGE2_EPOCHS = 15         # Hard negative refinement
LR = 1e-4                  # Single learning rate
WARMUP_EPOCHS = 3
MAX_GRAD_NORM = 1.0

# Stage 1 loss weights (focus on gap closing)
STAGE1_WEIGHTS = {
    'lambda_atp': 0.5,       # Align True Pairs
    'lambda_cu': 0.3,        # Centroid Uniformity
    'lambda_code': 0.3,
    'lambda_diversity': 0.1,
    'lambda_hard_neg': 0.0,  # No hard negatives in Stage 1
}

# Stage 2 loss weights (add hard negatives)
STAGE2_WEIGHTS = {
    'lambda_atp': 0.3,       # Reduce alignment weight
    'lambda_cu': 0.2,
    'lambda_code': 0.2,
    'lambda_diversity': 0.1,
    'lambda_hard_neg': 0.3,  # Enable hard negatives
}

print("Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Stage 1: {STAGE1_EPOCHS} epochs (gap closing)")
print(f"  Stage 2: {STAGE2_EPOCHS} epochs (hard negatives)")
print(f"  LR: {LR}")
print(f"  Output: {OUTPUT_DIR}")

Configuration:
  Data root: d:\Defect_Det\MMCAD\data
  Batch size: 256
  Stage 1: 20 epochs (gap closing)
  Stage 2: 15 epochs (hard negatives)
  LR: 0.0001
  Output: ..\outputs\gfa_v4_8


In [3]:
# Cell 3: Import Dataset
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

print("Using GFAMappedDataset with use_autobrep=True")
print("This loads:")
print("  - BRep features with topology (edge_to_faces, bfs_level, face_centroids)")
print("  - PC features from ShapeLLM")
print("  - Text features from Phi-4")
print()
print("Required files:")
print(f"  - {EMBEDDINGS_DIR / 'train_brep_autobrep.h5'}")
print(f"  - {EMBEDDINGS_DIR / 'val_brep_autobrep.h5'}")
print(f"  - {ALIGNED_DIR / 'uid_mapping_autobrep.json'}")
print(f"  - {PC_FILE}")

Using GFAMappedDataset with use_autobrep=True
This loads:
  - BRep features with topology (edge_to_faces, bfs_level, face_centroids)
  - PC features from ShapeLLM
  - Text features from Phi-4

Required files:
  - d:\Defect_Det\MMCAD\data\embeddings\train_brep_autobrep.h5
  - d:\Defect_Det\MMCAD\data\embeddings\val_brep_autobrep.h5
  - d:\Defect_Det\MMCAD\data\aligned\uid_mapping_autobrep.json
  - c:\Users\User\Desktop\pc_embeddings_full.h5


In [4]:
# Cell 4: Load Datasets
gc.collect()
torch.cuda.empty_cache()

print("Loading datasets with AutoBrep topology...")
print("=" * 60)

MAX_TRAIN_SAMPLES = 111000

# Train dataset - LOAD TO RAM
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset - ON DISK
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\n" + "=" * 60)
print("DATASETS READY!")

Loading datasets with AutoBrep topology...

[1/2] Loading TRAIN dataset (max 111,000 samples)...
  Filtered to 132324 samples with AutoBrep data (from 133105)
  Limiting to 111000 samples (from 132324)
  Loading train data to memory (B-Rep + PC + Text)...
    Loading AutoBrep with topology...
    Loading 111000 samples from train_brep_autobrep.h5...
      Done (111000 samples)
    AutoBrep loaded: 9.2GB in 138.2s
    Loading PC (50GB)...
    Loading Text from: c:\Users\User\Desktop\text_splits\train_text_embeddings.h5
    ✓ Text loaded: 174.6GB in 384.2s
  ✓ Loaded 111000 samples: 205.6GB in RAM (B-Rep + PC + Text)
GFAMappedDataset: train with 111000 samples (in memory)
Train: 111,000 samples

[2/2] Loading VAL dataset...
  Filtered to 16535 samples with AutoBrep data (from 16638)
GFAMappedDataset: val with 16535 samples
Val: 16,535 samples

DATASETS READY!


In [5]:
# Cell 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

# Verify sample has topology fields
sample = train_dataset[0]
print(f"\nSample keys: {list(sample.keys())}")
if 'edge_to_faces' in sample:
    print(f"  edge_to_faces: {sample['edge_to_faces'].shape}")
    print(f"  bfs_level: {sample['bfs_level'].shape}")
    print(f"  face_centroids: {sample['face_centroids'].shape}")
else:
    print("  WARNING: Topology fields not found!")

Train loader: 433 batches
Val loader: 65 batches

Sample keys: ['sample_id', 'idx', 'rot_idx', 'brep_face_features', 'brep_edge_features', 'brep_face_mask', 'brep_edge_mask', 'pc_features', 'edge_to_faces', 'bfs_level', 'face_centroids', 'face_normals', 'face_areas', 'edge_midpoints', 'edge_lengths', 'desc_embedding', 'desc_mask', 'has_brep', 'use_cached_brep_features', 'has_pointcloud', 'use_cached_pc_features', 'has_text', 'use_cached_embeddings']
  edge_to_faces: torch.Size([512, 2])
  bfs_level: torch.Size([192])
  face_centroids: torch.Size([192, 3])


In [6]:
# Cell 6: Batch Remapping Function
# ═══════════════════════════════════════════════════════════════════════════════
# The collate function outputs keys with 'brep_' prefix and combined pc_features.
# The model expects keys without prefix and split PC features.
# ═══════════════════════════════════════════════════════════════════════════════

def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        # Remove 'brep_' prefix for model compatibility
        if k.startswith('brep_'):
            new_key = k[5:]  # Remove 'brep_' prefix
            remapped[new_key] = v
        else:
            remapped[k] = v
    
    # Split pc_features into local and global
    if 'pc_features' in remapped:
        pc = remapped.pop('pc_features')
        # pc is (B, N+1, 1024) where last token is global
        remapped['pc_local_features'] = pc[:, :-1, :]   # (B, N, 1024)
        remapped['pc_global_features'] = pc[:, -1, :]   # (B, 1024)
    
    # Rename text keys
    if 'desc_embedding' in remapped:
        remapped['text_features'] = remapped.pop('desc_embedding')
    if 'desc_mask' in remapped:
        remapped['text_mask'] = remapped.pop('desc_mask')
    
    return remapped

# Test remapping
test_batch = next(iter(train_loader))
test_batch = remap_batch(test_batch)
print("Remapped batch keys:")
for k, v in test_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

Remapped batch keys:
  has_brep: torch.Size([256])
  has_pointcloud: torch.Size([256])
  has_text: torch.Size([256])
  face_features: torch.Size([256, 192, 48])
  edge_features: torch.Size([256, 512, 12])
  face_mask: torch.Size([256, 192])
  edge_mask: torch.Size([256, 512])
  edge_to_faces: torch.Size([256, 512, 2])
  bfs_level: torch.Size([256, 192])
  face_centroids: torch.Size([256, 192, 3])
  face_normals: torch.Size([256, 192, 3])
  face_areas: torch.Size([256, 192])
  edge_midpoints: torch.Size([256, 512, 3])
  edge_lengths: torch.Size([256, 512])
  idx: torch.Size([256])
  rot_idx: torch.Size([256])
  pc_local_features: torch.Size([256, 47, 1024])
  pc_global_features: torch.Size([256, 1024])
  text_features: torch.Size([256, 256, 3072])
  text_mask: torch.Size([256, 256])


In [7]:
# Cell 7: Create Model
gc.collect()
torch.cuda.empty_cache()

from clip4cad.models.clip4cad_gfa_v4_8 import CLIP4CAD_GFA_v48, GFAv48Config
from clip4cad.losses.gfa_v4_8_losses import (
    GFAv48Loss,
    compute_modality_gap,
    compute_true_pair_cosine,
    compute_code_diversity,
    mine_hard_negatives_by_code,
)

# Model configuration
config = GFAv48Config(
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    d_unified=256,
    d_proj=128,
    # Codebook
    num_codes=256,
    # Topology encoder
    num_msg_layers=3,
    num_brep_tf_layers=4,
    # Grounding
    num_heads=8,
    dropout=0.1,
)

model = CLIP4CAD_GFA_v48(config).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

c:\Users\User\anaconda3\envs\clip4cad\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Model parameters: 9,011,330
Trainable parameters: 9,011,330


In [8]:
# Cell 8: Initialize Codebook with K-Means
print("Initializing codebook with K-Means from text encoder outputs...")
print("=" * 60)

model.initialize_codebook(train_loader, device, remap_fn=remap_batch, max_batches=50)

print("\nCodebook initialized!")
print(f"  Codes shape: {model.codebook.codes.shape}")
print(f"  Temperature: {model.codebook.tau.item():.4f}")

Initializing codebook with K-Means from text encoder outputs...
Initializing codebook from text encoder outputs...


c:\Users\User\anaconda3\envs\clip4cad\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Initialized 256 codes from 10240 samples

Codebook initialized!
  Codes shape: torch.Size([256, 256])
  Temperature: 1.1000


In [9]:
# Cell 9: Training Setup

# Loss - simpler than v4.4, no curriculum!
criterion = GFAv48Loss(**STAGE1_WEIGHTS)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

# Scaler for FP16
scaler = GradScaler()

# Scheduler - cosine with warmup
total_epochs = STAGE1_EPOCHS + STAGE2_EPOCHS
warmup_steps = WARMUP_EPOCHS * len(train_loader)
total_steps = total_epochs * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(1e-6 / LR, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

print("Training setup complete!")
print(f"  Loss: GFAv48Loss (5 terms)")
print(f"  Optimizer: AdamW (lr={LR})")
print(f"  Scheduler: Cosine with {WARMUP_EPOCHS} epoch warmup")
print(f"  Total epochs: {total_epochs}")

Training setup complete!
  Loss: GFAv48Loss (5 terms)
  Optimizer: AdamW (lr=0.0001)
  Scheduler: Cosine with 3 epoch warmup
  Total epochs: 35


C:\Users\User\AppData\Local\Temp\ipykernel_5740\3608905724.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [10]:
# Cell 10: Verify Forward Pass Before Training
print("Verifying forward pass...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast():
        outputs = model(test_batch)

print("\nOutputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    else:
        print(f"  {k}: {v}")

# Compute initial metrics (using final embeddings z_*)
gap_brep, gap_pc = compute_modality_gap(
    outputs['z_text'], outputs['z_brep'], outputs['z_pc']
)
cos_brep, cos_pc = compute_true_pair_cosine(
    outputs['z_text'], outputs['z_brep'], outputs['z_pc']
)
diversity = compute_code_diversity(
    outputs['w_text'], outputs['w_brep'], outputs['w_pc']
)

print("\nInitial Metrics:")
print(f"  Gap (BRep): {gap_brep:.4f}")
print(f"  Gap (PC):   {gap_pc:.4f}")
print(f"  True-pair cos (BRep): {cos_brep:.4f}")
print(f"  True-pair cos (PC):   {cos_pc:.4f}")
print(f"  Code diversity: {diversity:.4f}")

# Test loss computation
loss, loss_dict = criterion(outputs)
print("\nLoss components:")
for k, v in loss_dict.items():
    print(f"  {k}: {v.item():.4f}")

Verifying forward pass...


C:\Users\User\AppData\Local\Temp\ipykernel_5740\3554323764.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Outputs:
  z_text: torch.Size([256, 128]) [OK]
  z_brep: torch.Size([256, 128]) [OK]
  z_pc: torch.Size([256, 128]) [OK]
  H_text: torch.Size([256, 256, 256]) [OK]
  H_brep: torch.Size([256, 256, 256]) [OK]
  H_pc: torch.Size([256, 256, 256]) [OK]
  w_text: torch.Size([256, 256]) [OK]
  w_brep: torch.Size([256, 256]) [OK]
  w_pc: torch.Size([256, 256]) [OK]
  G_text: torch.Size([256, 256, 256]) [OK]
  G_brep: torch.Size([256, 256, 704]) [OK]
  G_pc: torch.Size([256, 256, 48]) [OK]
  tau: torch.Size([]) [OK]

Initial Metrics:
  Gap (BRep): 2.4043
  Gap (PC):   2.4141
  True-pair cos (BRep): 0.0338
  True-pair cos (PC):   0.0980
  Code diversity: 1.0000

Loss components:
  contrastive: 5.9054
  atp: 7.5664
  cu: -1.3809
  code: -0.0000
  diversity: 0.0000
  hard_neg: 0.0000
  total: 9.2743


In [11]:
# Cell 11: Training Loop

global_step = 0
best_val_loss = float('inf')
best_gap = float('inf')
current_stage = 1
hard_negatives = None

history = {
    'train_loss': [], 'val_loss': [],
    'gap_brep': [], 'gap_pc': [],
    'cos_brep': [], 'cos_pc': [],
    'diversity': [],
    'contrastive': [], 'atp': [], 'cu': [], 'code': [],
}

print("Starting GFA v4.8 Training...")
print("=" * 70)
print("No curriculum learning - just gap closing with SharedFusionNetwork!")
print("=" * 70)

for epoch in range(1, total_epochs + 1):
    
    # ─────────────────────────────────────────────────────────────────────────
    # STAGE TRANSITION
    # ─────────────────────────────────────────────────────────────────────────
    if epoch == STAGE1_EPOCHS + 1:
        print("\n" + "=" * 70)
        print("TRANSITIONING TO STAGE 2 - HARD NEGATIVES")
        print("=" * 70)
        current_stage = 2
        
        # Update loss weights
        criterion.update_weights(**STAGE2_WEIGHTS)
        print(f"Updated loss weights: lambda_hard_neg={STAGE2_WEIGHTS['lambda_hard_neg']}")
        
        # Mine hard negatives based on code activation
        print("\nMining hard negatives by code activation...")
        hard_negatives = mine_hard_negatives_by_code(
            model, train_loader, device, top_k=10, max_batches=50
        )
        print(f"Mined {len(hard_negatives)} hard negative sets")
        
        # Save Stage 1 checkpoint
        torch.save({
            'epoch': epoch - 1,
            'model_state_dict': model.state_dict(),
            'best_gap': best_gap,
            'history': history,
        }, OUTPUT_DIR / 'checkpoint_stage1_final.pt')
    
    # ─────────────────────────────────────────────────────────────────────────
    # TRAIN EPOCH
    # ─────────────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'gap_brep': [], 'gap_pc': [],
        'cos_brep': [], 'cos_pc': [],
        'diversity': [],
        'contrastive': [], 'atp': [], 'cu': [], 'code': [],
    }
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} (Stage {current_stage})")
    
    for batch_idx, batch in enumerate(pbar):
        batch = remap_batch(batch)
        
        with autocast():
            outputs = model(batch)
            
            # Get hard negatives for this batch if in Stage 2
            batch_hard_negs = None
            if hard_negatives is not None:
                batch_size = outputs['z_brep'].shape[0]
                start_idx = batch_idx * batch_size
                batch_hard_negs = [
                    hard_negatives.get(start_idx + i) for i in range(batch_size)
                ]
            
            loss, loss_dict = criterion(outputs, hard_negatives=batch_hard_negs)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        global_step += 1
        epoch_loss += loss_dict['total'].item()
        
        # Compute metrics (using final embeddings z_*)
        with torch.no_grad():
            gap_brep, gap_pc = compute_modality_gap(
                outputs['z_text'], outputs['z_brep'], outputs['z_pc']
            )
            cos_brep, cos_pc = compute_true_pair_cosine(
                outputs['z_text'], outputs['z_brep'], outputs['z_pc']
            )
            diversity = compute_code_diversity(
                outputs['w_text'], outputs['w_brep'], outputs['w_pc']
            )
            
            epoch_metrics['gap_brep'].append(gap_brep)
            epoch_metrics['gap_pc'].append(gap_pc)
            epoch_metrics['cos_brep'].append(cos_brep)
            epoch_metrics['cos_pc'].append(cos_pc)
            epoch_metrics['diversity'].append(diversity)
            epoch_metrics['contrastive'].append(loss_dict['contrastive'].item())
            epoch_metrics['atp'].append(loss_dict['atp'].item())
            epoch_metrics['cu'].append(loss_dict['cu'].item())
            epoch_metrics['code'].append(loss_dict['code'].item())
        
        # Progress bar
        postfix = {
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{gap_brep:.2f}",
            'cos': f"{cos_brep:.3f}",
            'div': f"{diversity:.3f}",
        }
        pbar.set_postfix(postfix)
    
    # ─────────────────────────────────────────────────────────────────────────
    # EPOCH SUMMARY
    # ─────────────────────────────────────────────────────────────────────────
    avg_loss = epoch_loss / len(train_loader)
    avg_gap_brep = np.mean(epoch_metrics['gap_brep'])
    avg_gap_pc = np.mean(epoch_metrics['gap_pc'])
    avg_cos_brep = np.mean(epoch_metrics['cos_brep'])
    avg_cos_pc = np.mean(epoch_metrics['cos_pc'])
    avg_diversity = np.mean(epoch_metrics['diversity'])
    
    history['train_loss'].append(avg_loss)
    history['gap_brep'].append(avg_gap_brep)
    history['gap_pc'].append(avg_gap_pc)
    history['cos_brep'].append(avg_cos_brep)
    history['cos_pc'].append(avg_cos_pc)
    history['diversity'].append(avg_diversity)
    history['contrastive'].append(np.mean(epoch_metrics['contrastive']))
    history['atp'].append(np.mean(epoch_metrics['atp']))
    history['cu'].append(np.mean(epoch_metrics['cu']))
    history['code'].append(np.mean(epoch_metrics['code']))
    
    avg_gap = (avg_gap_brep + avg_gap_pc) / 2
    if avg_gap < best_gap:
        best_gap = avg_gap
    
    print(f"\nEpoch {epoch}: Loss={avg_loss:.4f}")
    print(f"  Gap: BRep={avg_gap_brep:.4f}, PC={avg_gap_pc:.4f} (best={best_gap:.4f})")
    print(f"  True-pair cos: BRep={avg_cos_brep:.4f}, PC={avg_cos_pc:.4f}")
    print(f"  Code diversity: {avg_diversity:.4f}")
    print(f"  Loss components: con={np.mean(epoch_metrics['contrastive']):.4f}, "
          f"atp={np.mean(epoch_metrics['atp']):.4f}, "
          f"cu={np.mean(epoch_metrics['cu']):.4f}, "
          f"code={np.mean(epoch_metrics['code']):.4f}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # VALIDATION (every 5 epochs)
    # ─────────────────────────────────────────────────────────────────────────
    if epoch % 5 == 0:
        model.eval()
        val_loss = 0.0
        val_metrics = {'gap_brep': [], 'cos_brep': [], 'diversity': []}
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                batch = remap_batch(batch)
                with autocast():
                    outputs = model(batch)
                    loss, loss_dict = criterion(outputs)
                val_loss += loss_dict['total'].item()
                
                gap_brep, _ = compute_modality_gap(
                    outputs['z_text'], outputs['z_brep'], outputs['z_pc']
                )
                cos_brep, _ = compute_true_pair_cosine(
                    outputs['z_text'], outputs['z_brep'], outputs['z_pc']
                )
                diversity = compute_code_diversity(
                    outputs['w_text'], outputs['w_brep'], outputs['w_pc']
                )
                
                val_metrics['gap_brep'].append(gap_brep)
                val_metrics['cos_brep'].append(cos_brep)
                val_metrics['diversity'].append(diversity)
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_gap = np.mean(val_metrics['gap_brep'])
        avg_val_cos = np.mean(val_metrics['cos_brep'])
        avg_val_div = np.mean(val_metrics['diversity'])
        
        history['val_loss'].append(avg_val_loss)
        print(f"Validation: Loss={avg_val_loss:.4f}, Gap={avg_val_gap:.4f}, "
              f"Cos={avg_val_cos:.4f}, Div={avg_val_div:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_loss': best_val_loss,
                'best_gap': best_gap,
            }, OUTPUT_DIR / 'checkpoint_best.pt')
            print("Saved best model!")
    
    # Save periodic checkpoint
    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_gap': best_gap,
            'history': history,
        }, OUTPUT_DIR / f'checkpoint_epoch{epoch}.pt')

print("\n" + "=" * 70)
print("Training Complete!")
print(f"Best gap: {best_gap:.4f}")
print(f"Best val loss: {best_val_loss:.4f}")

Starting GFA v4.8 Training...
No curriculum learning - just gap closing with SharedFusionNetwork!


Epoch 1 (Stage 1):   0%|          | 0/433 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Temp\ipykernel_5740\2818317425.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\User\anaconda3\envs\clip4cad\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(



Epoch 1: Loss=9.2930
  Gap: BRep=2.4115, PC=2.3915 (best=2.4015)
  True-pair cos: BRep=0.0345, PC=0.1067
  Code diversity: 1.0000
  Loss components: con=5.9230, atp=7.5577, cu=-1.3631, code=-0.0000


Epoch 2 (Stage 1):   0%|          | 0/433 [00:00<?, ?it/s]


Epoch 2: Loss=9.2927
  Gap: BRep=2.4115, PC=2.3915 (best=2.4015)
  True-pair cos: BRep=0.0345, PC=0.1067
  Code diversity: 1.0000
  Loss components: con=5.9228, atp=7.5576, cu=-1.3631, code=-0.0000


Epoch 3 (Stage 1):   0%|          | 0/433 [00:00<?, ?it/s]


Epoch 3: Loss=9.2928
  Gap: BRep=2.4115, PC=2.3914 (best=2.4014)
  True-pair cos: BRep=0.0345, PC=0.1067
  Code diversity: 1.0000
  Loss components: con=5.9231, atp=7.5573, cu=-1.3631, code=-0.0000


Epoch 4 (Stage 1):   0%|          | 0/433 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Cell 12: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train')
if history['val_loss']:
    val_epochs = list(range(5, len(history['train_loss']) + 1, 5))
    axes[0, 0].plot(val_epochs, history['val_loss'], 'o-', label='Val')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Modality Gap (KEY METRIC)
axes[0, 1].plot(history['gap_brep'], label='BRep')
axes[0, 1].plot(history['gap_pc'], label='PC')
axes[0, 1].axhline(y=0, color='g', linestyle='--', alpha=0.5, label='Target (0)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Gap')
axes[0, 1].set_title('Modality Gap (should → 0)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# True-Pair Cosine (KEY METRIC)
axes[0, 2].plot(history['cos_brep'], label='BRep')
axes[0, 2].plot(history['cos_pc'], label='PC')
axes[0, 2].axhline(y=1.0, color='g', linestyle='--', alpha=0.5, label='Target (1.0)')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Cosine Similarity')
axes[0, 2].set_title('True-Pair Cosine (should → 1.0)')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Code Diversity
axes[1, 0].plot(history['diversity'])
axes[1, 0].axhline(y=0.8, color='g', linestyle='--', alpha=0.5, label='Target (>0.8)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Diversity')
axes[1, 0].set_title('Code Diversity')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Loss Components
axes[1, 1].plot(history['contrastive'], label='Contrastive')
axes[1, 1].plot(history['atp'], label='ATP')
axes[1, 1].plot(history['cu'], label='CU')
axes[1, 1].plot(history['code'], label='Code')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Loss Components')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Stage transition marker
for ax in axes.flat:
    ax.axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', alpha=0.3)

axes[1, 2].axis('off')
axes[1, 2].text(0.5, 0.5, 
    f"GFA v4.8 Summary\n\n"
    f"Best Gap: {best_gap:.4f}\n"
    f"Best Val Loss: {best_val_loss:.4f}\n\n"
    f"Final Metrics:\n"
    f"  Gap BRep: {history['gap_brep'][-1]:.4f}\n"
    f"  Gap PC: {history['gap_pc'][-1]:.4f}\n"
    f"  Cos BRep: {history['cos_brep'][-1]:.4f}\n"
    f"  Cos PC: {history['cos_pc'][-1]:.4f}\n"
    f"  Diversity: {history['diversity'][-1]:.4f}",
    ha='center', va='center', fontsize=12, family='monospace',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 13: Save Final Model
torch.save({
    'config': config,
    'model_state_dict': model.state_dict(),
    'best_gap': best_gap,
    'best_val_loss': best_val_loss,
    'history': history,
}, OUTPUT_DIR / 'gfa_v4_8_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'gfa_v4_8_final.pt'}")
print(f"Best modality gap: {best_gap:.4f}")
print(f"Best val loss: {best_val_loss:.4f}")

In [ ]:
# Cell 14: Quick Retrieval Test
print("Testing retrieval performance...")

model.eval()
all_z_text = []
all_z_brep = []
all_z_pc = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast():
            outputs = model(batch)
        
        all_z_text.append(outputs['z_text'].cpu())
        all_z_brep.append(outputs['z_brep'].cpu())
        all_z_pc.append(outputs['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

# Normalize
z_text = F.normalize(z_text, dim=-1)
z_brep = F.normalize(z_brep, dim=-1)
z_pc = F.normalize(z_pc, dim=-1)

N = z_text.shape[0]
print(f"\nEvaluation set: {N} samples")

# Text → BRep retrieval
sim_t2b = z_text @ z_brep.T  # (N, N)
ranks_t2b = (sim_t2b.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_t2b = (ranks_t2b < 1).float().mean().item() * 100
r5_t2b = (ranks_t2b < 5).float().mean().item() * 100
r10_t2b = (ranks_t2b < 10).float().mean().item() * 100

# Text → PC retrieval
sim_t2p = z_text @ z_pc.T
ranks_t2p = (sim_t2p.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_t2p = (ranks_t2p < 1).float().mean().item() * 100
r5_t2p = (ranks_t2p < 5).float().mean().item() * 100

# BRep → PC retrieval (geometry-only)
sim_b2p = z_brep @ z_pc.T
ranks_b2p = (sim_b2p.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1_b2p = (ranks_b2p < 1).float().mean().item() * 100
r5_b2p = (ranks_b2p < 5).float().mean().item() * 100

print("\n" + "=" * 50)
print("Retrieval Results:")
print("=" * 50)
print(f"Text → BRep: R@1={r1_t2b:.1f}%, R@5={r5_t2b:.1f}%, R@10={r10_t2b:.1f}%")
print(f"Text → PC:   R@1={r1_t2p:.1f}%, R@5={r5_t2p:.1f}%")
print(f"BRep → PC:   R@1={r1_b2p:.1f}%, R@5={r5_b2p:.1f}%")
print("=" * 50)